In [ ]:
path_prefix = '../..'

import sys
sys.path.append(path_prefix)

import json
import numpy as np
import pandas as pd

## Analysis

In [ ]:
dataset_path = f'{path_prefix}/data/phd/phd.json'
save_path = f'{path_prefix}/data/phd/prototype/phd_sample.json'
sample_size = None

with open(dataset_path, 'r', encoding='utf-8') as f:
    dataset = json.load(f)
if sample_size is not None and len(dataset) > sample_size:
    dataset = dataset[:sample_size]

df = pd.DataFrame(dataset)

# Drop css_description 
df = df[df['ccs_description'].isna()]
df = df.drop(columns=['ccs_description'])

print(f'Successfully load the Hallusion Bench dataset with: {len(df)} samples.')
df.head()

In [ ]:
task_counts = df['task'].value_counts()
task_counts

In [ ]:
yes_answer_nunique = df['yes_question'].nunique()
no_answer_nunique = df['no_question'].nunique()

print('Yes answer unique count:', yes_answer_nunique)
print('No answer unique count:', no_answer_nunique)

## Dataset Construction

In [ ]:
def sample_and_balance(group, sample_size=1500, seed=42):
    np.random.seed(seed)
    n_sample = min(sample_size, len(group))
    
    # Create unique identifier excluding unhashable columns
    hashable_cols = ['yes_question', 'no_question']
    group['unique_id'] = range(len(group))  # Temporary unique ID
    
    # Sample maximizing unique yes/no question pairs
    unique_pairs = group.drop_duplicates(subset=hashable_cols)
    n_unique = min(n_sample, len(unique_pairs))
    group_sampled = unique_pairs.sample(n=n_unique, random_state=seed)
    
    # Fill remaining if needed (no drop_duplicates to avoid dict error)
    remaining_needed = n_sample - len(group_sampled)
    if remaining_needed > 0:
        additional = group.sample(n=remaining_needed, random_state=seed)
        group_sampled = pd.concat([group_sampled, additional]).reset_index(drop=True)
    group_sampled = group_sampled.drop(columns=['unique_id'], errors='ignore')
    
    # Balance yes/no labels
    n = len(group_sampled)
    yes_indices = np.random.choice(group_sampled.index, size=n//2, replace=False)
    no_indices = np.setdiff1d(group_sampled.index, yes_indices)
    
    group_sampled.loc[yes_indices, 'question'] = group_sampled.loc[yes_indices, 'yes_question']
    group_sampled.loc[no_indices, 'question'] = group_sampled.loc[no_indices, 'no_question']
    group_sampled['label'] = 0
    group_sampled.loc[yes_indices, 'label'] = 1
    
    return group_sampled

In [ ]:
df_sample = (df
             .groupby('task', group_keys=False)
             .apply(sample_and_balance)
             .reset_index(drop=True))

print(f'Shape: {df_sample.shape}')
print(f'Number of unique questions: {df_sample['question'].nunique()}')
print(f'Number of unique yes questions: {df_sample['yes_question'].nunique()}')
print(f'Number of unique no questoins: {df_sample['no_question'].nunique()}')
print('\n', df_sample['label'].value_counts(normalize=True).round(3))
print('\n', df_sample['task'].value_counts(normalize=True).round(3))
print('\n', df_sample.groupby('task')['label'].value_counts(normalize=True).round(3).unstack(fill_value=0))
df_sample.head()

In [ ]:
df_sample = df_sample.reset_index().rename(columns={'index': 'id'})
output = df_sample.to_dict('records')

with open(save_path, 'w') as f:
    json.dump(output, f, indent=4)